# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** · Meridiano Imobiliária

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu a base **como ela
chega**: 2.930 registros, 82 colunas, 27 delas com célula vazia, escalas de qualidade
guardadas como texto, dois anos impossíveis e cinco imóveis que o próprio autor da base
recomenda descartar. Este notebook é o passo seguinte: **aplicar as correções** e gravar
a base curada que todas as fases posteriores consomem.

A regra deste notebook é uma só:

> **Nenhum dado é alterado em silêncio.** Cada ajuste declara o *motivo*, mostra a
> *evidência numérica* que o justifica e registra *quantos registros* foram tocados. O
> conjunto vai para `reports/tables/ajustes-dados/registro-de-ajustes.csv` e é publicado
> na documentação.

Isso vale também para o que é descartado: coluna que sai da base e registro que sai da
carteira aparecem no mesmo registro, com a ação `removido` e o critério ao lado.

## Duas regras de corte, declaradas antes de começar

O notebook não decide caso a caso o que imputar e o que descartar. Duas regras respondem
por todos os cortes, e as duas estão escritas antes de o primeiro dado ser tocado:

> **Regra 1 — ausência acima de 5% não se imputa, se descarta.** Vale para coluna e para
> registro. Acima desse ponto a imputação deixa de completar o dado e passa a produzi-lo:
> imputar 16,7% de uma coluna pela mediana significa que um sexto dela foi fabricado pelo
> próprio método, com a variância artificialmente reduzida em torno do valor imputado.
>
> **Regra 2 — o que não descreve um imóvel residencial comparável sai da carteira.**
> Valor extremo, venda que não é normal, cadastro implausível e coluna em que 99% dos
> imóveis dizem a mesma coisa.

A Regra 1 tem uma condição de aplicação que não é detalhe: ela roda **depois** da
ausência estrutural virar rótulo. Antes disso, um imóvel sem piscina, sem viela e sem
cerca teria 17% de campos vazios e seria descartado por não ter aquilo que nunca existiu.

## Ordem das operações

A sequência não é arbitrária — cada passo depende do anterior:

| # | Bloco | Por que vem nesta posição |
|:--|:---|:---|
| 1 | Leitura e sentinelas | Antes de contar ausência é preciso saber o que a origem chama de ausente |
| 2 | Duplicidade | Registro repetido pesa duas vezes em qualquer estatística usada adiante |
| 3 | Ausência estrutural | Separa "não tem" de "não sei" — só o segundo grupo entra na Regra 1 |
| 4 | Inconsistências internas | Corrigir o ano impossível antes de derivar idade a partir dele |
| 5 | Corte por ausência e imputação | Regra 1: descarta acima de 5%, imputa o resto, que fica abaixo de 1% |
| 6 | Escalas ordinais | `Ex > Gd > TA` vira inteiro depois que a ausência já virou rótulo |
| 7 | Renomeação | Da coluna com espaço no meio ao `snake_case` em português |
| 8 | Atributos derivados | Dependem das colunas já limpas, imputadas e renomeadas |
| 9 | Registros acima de 4.000 pés² | Recomendação do autor da base, aplicada sobre a base já derivada |
| 10 | Depuração da carteira | Regra 2: extremos, vendas não normais, cadastros implausíveis e colunas sem informação |

## Saída

* `data/processed/imoveis_curados.parquet` — a base curada, em **unidades originais**,
  legível por gente de negócio;
* `data/processed/manifesto_curadoria.json` — o que entrou, o que saiu e quantos ajustes;
* `reports/tables/ajustes-dados/` — o registro de ajustes e as tabelas de evidência;
* `docs/imagens/ajustes/` — as figuras.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.data import ingestao, preparacao, qualidade
from src.eda import estatistica as est
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

# As figuras deste notebook vão para `docs/imagens/ajustes/`; as tabelas, para
# `reports/tables/ajustes-dados/`. A regra do projeto é que nenhuma figura existe
# sem a tabela que a produziu.
FIGURAS = config.DOCS / "imagens" / "ajustes"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "ajustes-dados"

# Cores da identidade do projeto (docs/assets/css/brand.css).
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/ajustes/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


print(f"Projeto em {RAIZ}")

## O registro de ajustes

Toda alteração passa por `registrar`. A função não altera dado nenhum — ela só anota o
que foi feito, por quê e sobre quantos registros. O que não está no registro não
aconteceu, e o que aconteceu sem estar no registro é bug.

O campo `acao` distingue quatro desfechos:

| `acao` | Significado |
|:---|:---|
| `corrigido` | O valor mudou |
| `imputado` | A célula estava vazia e passou a ter valor |
| `removido` | O registro saiu da base |
| `verificado, sem ação` | O padrão foi conferido e a decisão foi **não** mexer |

E o campo `unidade` diz **o que** foi contado. Renomear atinge colunas, imputar atinge
células, remover atinge registros — somar tudo como "registro afetado" inflaria o volume
e faria o resumo mentir.

In [ ]:
AJUSTES: list[dict] = []


def pt(numero) -> str:
    """Formata o número no padrão brasileiro: ponto como separador de milhar."""
    return f"{numero:,}".replace(",", ".")


def registrar(
    bloco: str,
    ajuste: str,
    motivo: str,
    afetados: int,
    unidade: str = "registros",
    acao: str = "corrigido",
):
    """Anota um ajuste no registro, com motivo, desfecho e volume afetado.

    A `unidade` importa: renomear coluna afeta *colunas*, imputar afeta *células*
    e remover afeta *registros*. Contar tudo como registro infla o volume e faz o
    resumo mentir.
    """
    AJUSTES.append(
        {
            "n": len(AJUSTES) + 1,
            "bloco": bloco,
            "ajuste": ajuste,
            "motivo": motivo,
            "afetados": int(afetados),
            "unidade": unidade,
            "acao": acao,
        }
    )
    rotulo = unidade if int(afetados) != 1 else unidade.rstrip("s")
    print(f"{len(AJUSTES):>2}. [{acao:>20}] {ajuste} — {pt(int(afetados))} {rotulo}")

# 1 · Leitura — as sentinelas de ausência

> A camada bruta em `data/raw/` é **imutável**: nada neste notebook escreve nela. A
> primeira cópia em memória é o ponto de partida de todos os ajustes.

O primeiro ajuste acontece antes de qualquer linha ser lida: **o que a origem chama de
ausente**. O arquivo de De Cock usa `NA` para "não se aplica" e a palavra `None` como
categoria legítima de `Mas Vnr Type` — "sem revestimento em alvenaria aparente".

A leitura padrão do pandas converte as duas em nulo. O resultado é que 1.752 imóveis
**descritos** viram 1.752 imóveis **desconhecidos**, e a ausência da coluna salta de
0,8% para 60,6%. Por isso a ingestão declara `na_values=["NA", ""]` explicitamente
(`src/data/ingestao.py`), e o inventário abaixo é a evidência de que a decisão veio do
arquivo, não do hábito.

In [ ]:
# Carregando a base bruta — o parquet preserva os tipos e lê ~50× mais rápido que o
# .txt original; o texto fica como plano B caso a ingestão ainda não tenha rodado.
if config.ARQUIVO_BRUTO_PARQUET.exists():
    bruto = pd.read_parquet(config.ARQUIVO_BRUTO_PARQUET)
    origem = config.ARQUIVO_BRUTO_PARQUET
else:
    bruto = pd.read_csv(
        config.ARQUIVO_BRUTO, sep="\t", keep_default_na=False, na_values=ingestao.SENTINELAS
    )
    origem = config.ARQUIVO_BRUTO

# A partir daqui trabalha-se sobre a cópia. `bruto` fica intacto para as comparações
# de antes e depois no fim do notebook.
dados = bruto.copy()

LINHAS_ENTRADA, COLUNAS_ENTRADA = dados.shape
print(f"{origem.relative_to(RAIZ)} — {pt(LINHAS_ENTRADA)} linhas × {COLUNAS_ENTRADA} colunas")
print(f"Células vazias na entrada: {pt(int(dados.isna().sum().sum()))}")

In [ ]:
# Ocorrências literais de `NA`, `None` e vazio, coluna a coluna, lidas do arquivo de
# texto sem nenhuma conversão. Só o `.txt` sustenta esta contagem: no parquet a
# decisão já está tomada.
if config.ARQUIVO_BRUTO.exists():
    sentinelas = ingestao.inventario_sentinelas()
    io.salvar_tabela(sentinelas.reset_index(), SECAO, "inventario-de-sentinelas", indice=False)

    categoria_legitima = int(sentinelas["None_literal"].sum())
    registrar(
        "1 · Leitura",
        "Ler o arquivo com `na_values=['NA', '']` em vez do padrão do pandas",
        "A palavra `None` em `Mas Vnr Type` é categoria — 'sem revestimento em alvenaria "
        "aparente'. Na leitura padrão ela vira nulo e a ausência da coluna passa de 0,8% "
        "para 60,6%.",
        categoria_legitima,
        unidade="registros descritos que virariam desconhecidos",
        acao="corrigido",
    )
    display(sentinelas.head(12))
else:
    print("Arquivo .txt ausente — inventário de sentinelas pulado.")

# 2 · Duplicidade — o mesmo imóvel contado duas vezes?

A EDA encontrou **duas linhas idênticas** nas 24 características do espaço de atributos.
Antes de remover é preciso responder: são o mesmo imóvel, ou dois imóveis parecidos?

Três contagens respondem:

1. **duplicidade completa** nas 82 colunas — nenhuma;
2. **duplicidade sem os identificadores** `Order` e `PID` — nenhuma;
3. **duplicidade no espaço de atributos** — dois pares.

O terceiro caso não é registro repetido: são **imóveis distintos que coincidem nas
características medidas**. A tabela abaixo mostra em que colunas cada par difere fora do
espaço de atributos, e a condição de venda explica o resto — `Alloca` significa, na
documentação da base, *duas matrículas ligadas vendidas juntas*, tipicamente as duas
metades de um duplex.

**Decisão: mantidos.** A base curada descreve imóveis, e são dois imóveis com escritura
própria. A questão volta na matriz de modelagem, onde eles ocupam o mesmo ponto do
espaço — e lá o efeito é de peso, não de duplicidade de cadastro.

In [ ]:
# Do nome curado de volta ao nome original da coluna, para reproduzir o recorte da EDA.
ORIGINAL = {curado: bruta for bruta, curado in dic.RENOMEAR.items()}

ESPACO = (
    dic.NOMINAIS
    + dic.ORDINAIS
    + dic.NUMERICAS
    + dic.ATRIBUTOS_TRANSACAO
    + [dic.RESERVADA_VALIDACAO, dic.ROTULO_RESERVADO]
)
RECORTE_EDA = [ORIGINAL[curado] for curado in ESPACO if curado in ORIGINAL] + ["Lot Frontage"]

contagens = {
    "linhas": len(dados),
    "duplicadas nas 82 colunas": int(dados.duplicated().sum()),
    "duplicadas sem Order e PID": int(dados.drop(columns=["Order", "PID"]).duplicated().sum()),
    f"duplicadas no espaço de atributos ({len(RECORTE_EDA)} colunas)": int(
        dados[RECORTE_EDA].duplicated().sum()
    ),
    "PID único": bool(dados["PID"].is_unique),
}

# Em que colunas cada par coincidente difere no conjunto completo.
# `astype(str)` mantém o vazio como ausente; o rótulo explícito evita que duas
# células vazias sejam lidas como valores diferentes na chave.
chave = dados[RECORTE_EDA].astype(str).fillna("NA").apply("|".join, axis=1)
repetidas = chave[chave.duplicated(keep=False)]

linhas = []
for _, grupo in repetidas.groupby(repetidas):
    a, b = grupo.index[0], grupo.index[1]
    difere = [
        coluna
        for coluna in dados.columns
        if coluna not in ("Order", "PID")
        and not (
            dados.loc[a, coluna] == dados.loc[b, coluna]
            or (pd.isna(dados.loc[a, coluna]) and pd.isna(dados.loc[b, coluna]))
        )
    ]
    linhas.append(
        {
            "order_a": dados.loc[a, "Order"],
            "order_b": dados.loc[b, "Order"],
            "pid_a": dados.loc[a, "PID"],
            "pid_b": dados.loc[b, "PID"],
            "bairro": dados.loc[a, "Neighborhood"],
            "tipo": dados.loc[a, "Bldg Type"],
            "condicao_venda": dados.loc[a, "Sale Condition"],
            "difere_em": ", ".join(difere) or "—",
        }
    )

pares_coincidentes = pd.DataFrame(linhas)
io.salvar_tabela(pares_coincidentes, SECAO, "pares-coincidentes", indice=False)

registrar(
    "2 · Duplicidade",
    "Manter os pares que coincidem no espaço de atributos",
    "Não são registros repetidos: têm PID próprio, diferem em colunas fora do espaço "
    "analisado e são vendas `Alloca` — duas matrículas ligadas, as duas metades de um "
    "duplex.",
    len(pares_coincidentes) * 2,
    unidade="registros",
    acao="verificado, sem ação",
)

display(pd.Series(contagens).to_frame("verificação"))
pares_coincidentes

# 3 · Ausência estrutural — o `NA` que não é falta de dado

27 das 82 colunas têm célula vazia, num total de 13.997 células. Tratar todas do mesmo
jeito seria o erro mais caro deste notebook: **em Ames, quase toda a ausência é
estrutural**. `NA` em `Pool QC` não é dado perdido, é imóvel sem piscina; imputar pela
mediana inventaria 2.917 piscinas.

O projeto declara as duas naturezas em `src/data/dicionario.py`:

* `AUSENCIA_ESTRUTURAL` — 14 colunas em que `NA` significa "o item não existe". O vazio
  vira **rótulo explícito** (`SemPorao`, `SemGaragem`, `SemPiscina`…), e a coluna deixa
  de ter nulo sem que nada tenha sido inventado;
* `AUSENCIA_DESCONHECIDA` — `Mas Vnr Type`, onde os 23 vazios restantes são ausência de
  fato e recebem o rótulo `NaoInformado`, que não se confunde com a categoria `None`.

O que sobra depois desses dois mapas é a ausência real, tratada no bloco 5.

In [ ]:
ausentes = (
    dados.isna()
    .sum()
    .rename("ausentes")
    .to_frame()
    .assign(percentual=lambda t: (100 * t.ausentes / len(dados)).round(2))
    .query("ausentes > 0")
    .sort_values("ausentes", ascending=False)
    .reset_index(names="variavel")
)
ausentes["natureza"] = np.where(
    ausentes.variavel.isin(dic.AUSENCIA_ESTRUTURAL),
    "estrutural",
    np.where(ausentes.variavel.isin(dic.AUSENCIA_DESCONHECIDA), "desconhecida declarada", "real"),
)

CORES_NATUREZA = {
    "estrutural": DESTAQUE,
    "desconhecida declarada": REFERENCIA,
    "real": ALERTA,
}

fig, ax = plt.subplots(figsize=(15, 6))
sns.barplot(
    data=ausentes, x="variavel", y="percentual", hue="natureza", palette=CORES_NATUREZA, ax=ax
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.1f%%", fontsize=8, padding=2)
ax.tick_params(axis="x", rotation=60)
ax.set_xlabel("Colunas com ao menos uma célula vazia")
ax.set_ylabel("% de células ausentes")
ax.set_title(
    f"Natureza da ausência antes dos ajustes — {len(ausentes)} de {dados.shape[1]} colunas"
)
ax.legend(title="Natureza")

salvar(fig, "ausencia-por-natureza", ausentes)

resumo_natureza = (
    ausentes.groupby("natureza")
    .agg(colunas=("variavel", "size"), celulas=("ausentes", "sum"))
    .sort_values("celulas", ascending=False)
)
display(resumo_natureza)
ausentes

In [ ]:
# Ausência estrutural: o vazio vira rótulo. Nada é imputado — o rótulo diz que o item
# não existe, que é exatamente o que a origem quis dizer com `NA`.
tratadas = []
for coluna, rotulo in dic.AUSENCIA_ESTRUTURAL.items():
    celulas = int(dados[coluna].isna().sum())
    if celulas:
        dados[coluna] = dados[coluna].fillna(rotulo)
    tratadas.append({"coluna": coluna, "rotulo": rotulo, "celulas": celulas})

for coluna, rotulo in dic.AUSENCIA_DESCONHECIDA.items():
    celulas = int(dados[coluna].isna().sum())
    if celulas:
        dados[coluna] = dados[coluna].fillna(rotulo)
    tratadas.append({"coluna": coluna, "rotulo": rotulo, "celulas": celulas})

rotulagem = pd.DataFrame(tratadas).sort_values("celulas", ascending=False)
io.salvar_tabela(rotulagem, SECAO, "rotulagem-de-ausencia", indice=False)

estruturais = int(rotulagem[rotulagem.coluna.isin(dic.AUSENCIA_ESTRUTURAL)].celulas.sum())
desconhecidas = int(rotulagem[rotulagem.coluna.isin(dic.AUSENCIA_DESCONHECIDA)].celulas.sum())

registrar(
    "3 · Ausência",
    "Converter `NA` em rótulo de ausência estrutural em 14 colunas",
    "`NA` ali significa 'o item não existe' — sem porão, sem garagem, sem lareira, sem "
    "piscina. Imputar pela mediana inventaria itens que nunca existiram.",
    estruturais,
    unidade="células",
    acao="corrigido",
)
registrar(
    "3 · Ausência",
    "Rotular como `NaoInformado` os vazios de `Mas Vnr Type`",
    "Sobram 23 vazios que não são a categoria `None` (sem revestimento) e nem ausência "
    "estrutural: são desconhecidos de fato, e o rótulo os separa dos 1.752 descritos.",
    desconhecidas,
    unidade="células",
    acao="corrigido",
)

print(f"Células vazias restantes: {pt(int(dados.isna().sum().sum()))}")
rotulagem

# 4 · Inconsistências internas

Três padrões que nenhuma contagem de nulos revela — só a checagem de plausibilidade do
domínio, em `src/data/qualidade.py`. São poucos registros, mas cada um contamina uma
variável derivada inteira.

| Inconsistência | Registros | Correção | Por quê |
|:---|--:|:---|:---|
| Garagem construída depois da venda | 2 | `ano_garagem = ano_construcao` | Um traz **2207** com venda em 2007: erro de digitação evidente. Sem corrigir, a idade da garagem fica **200 anos negativa** e domina qualquer padronização |
| Reforma anterior à construção | 1 | `ano_reforma = ano_construcao` | Casa construída em 2002 com reforma em 2001. Sem corrigir, o imóvel tem mais anos desde a reforma do que anos de vida |
| Garagem declarada sem nenhuma medida | 1 | `tipo_garagem = SemGaragem` | O registro traz `Detchd` e deixa área, vagas, ano e as duas notas em branco. As outras colunas do bloco já foram rotuladas `SemGaragem`; manter só o tipo produziria uma garagem de área zero |

A terceira é a mais sutil: não é valor errado, é **incoerência entre colunas**. Ela só
aparece porque a flag derivada `tem_garagem` é calculada da área, e o imóvel sairia com
`tipo_garagem = Detchd` e `tem_garagem = 0`.

Um registro continua marcando a primeira regra depois da correção, e não é falha: o
imóvel foi **construído em 2008 e vendido em 2007** — obra concluída depois da venda
registrada. Não há ano errado para corrigir, a casa inteira é posterior à transação. Ele
sai no bloco 9 por outro motivo, a área de 5.095 pés².

In [ ]:
# Antes: os registros que violam cada regra.
antes_garagem = dados.loc[
    dados["Garage Yr Blt"] > dados["Yr Sold"],
    ["Order", "Year Built", "Garage Yr Blt", "Yr Sold"],
].copy()
antes_reforma = dados.loc[
    dados["Year Remod/Add"] < dados["Year Built"],
    ["Order", "Year Built", "Year Remod/Add", "Yr Sold"],
].copy()

COLUNAS_GARAGEM = ["Garage Type", "Garage Yr Blt", "Garage Finish", "Garage Cars", "Garage Area"]
sem_medida = (
    dados["Garage Type"].ne(dic.AUSENCIA_ESTRUTURAL["Garage Type"])
    & dados["Garage Area"].isna()
    & dados["Garage Cars"].isna()
    & dados["Garage Yr Blt"].isna()
)
antes_incoerente = dados.loc[
    sem_medida, ["Order", "Year Built", "Yr Sold", *COLUNAS_GARAGEM]
].copy()

inconsistencias = pd.concat(
    [
        antes_garagem.assign(inconsistencia="garagem construída depois da venda"),
        antes_reforma.assign(inconsistencia="reforma anterior à construção"),
        antes_incoerente.assign(inconsistencia="garagem declarada sem medida"),
    ],
    ignore_index=True,
)
io.salvar_tabela(inconsistencias, SECAO, "inconsistencias-antes", indice=False)
inconsistencias

In [ ]:
# Correção 1 — o ano da garagem passa a ser o ano de construção da casa.
fora_de_faixa = dados["Garage Yr Blt"] > dados["Yr Sold"]
dados.loc[fora_de_faixa, "Garage Yr Blt"] = dados.loc[fora_de_faixa, "Year Built"]
registrar(
    "4 · Inconsistências",
    "Corrigir o ano da garagem posterior à venda (`2207` → ano de construção)",
    "Erro de digitação evidente: o imóvel foi vendido em 2007 e construído em 2006. Sem "
    "a correção a idade da garagem fica negativa em 200 anos.",
    int(fora_de_faixa.sum()),
)

# Correção 2 — a reforma não pode anteceder a construção.
reforma_invertida = dados["Year Remod/Add"] < dados["Year Built"]
dados.loc[reforma_invertida, "Year Remod/Add"] = dados.loc[reforma_invertida, "Year Built"]
registrar(
    "4 · Inconsistências",
    "Alinhar `ano_reforma` ao `ano_construcao` onde a ordem está invertida",
    "Imóvel construído em 2002 com reforma registrada em 2001. Reforma anterior à "
    "construção não existe; o alinhamento zera os anos desde a reforma, que é o que "
    "'nunca reformado' significa.",
    int(reforma_invertida.sum()),
)

# Correção 3 — garagem sem nenhuma medida deixa de ser garagem.
dados.loc[sem_medida, "Garage Type"] = dic.AUSENCIA_ESTRUTURAL["Garage Type"]
registrar(
    "4 · Inconsistências",
    "Converter `tipo_garagem` em `SemGaragem` no registro sem nenhuma medida",
    "Área, vagas, ano e as duas notas vêm todas vazias. Sem medida não há garagem para "
    "descrever, e manter o tipo produziria um imóvel com garagem de área zero.",
    int(sem_medida.sum()),
)

# Depois: as mesmas regras, agora sem violação.
verificacao = pd.DataFrame(
    [
        {
            "regra": "garagem construída depois da venda",
            "antes": len(antes_garagem),
            "depois": int((dados["Garage Yr Blt"] > dados["Yr Sold"]).sum()),
        },
        {
            "regra": "reforma anterior à construção",
            "antes": len(antes_reforma),
            "depois": int((dados["Year Remod/Add"] < dados["Year Built"]).sum()),
        },
        {
            "regra": "garagem declarada sem medida",
            "antes": len(antes_incoerente),
            "depois": int(
                (
                    dados["Garage Type"].ne(dic.AUSENCIA_ESTRUTURAL["Garage Type"])
                    & dados["Garage Area"].isna()
                    & dados["Garage Cars"].isna()
                ).sum()
            ),
        },
    ]
)
io.salvar_tabela(verificacao, SECAO, "inconsistencias-antes-e-depois", indice=False)

# O registro que sobra na primeira regra é o imóvel vendido na planta: a obra terminou
# em 2008 e a venda foi registrada em 2007. Não há erro de digitação a corrigir — a
# garagem foi construída depois da venda porque a casa também foi. Ele sai no bloco 9,
# por ter 5.095 pés² de área habitável.
pendente = dados.loc[
    dados["Garage Yr Blt"] > dados["Yr Sold"],
    ["Order", "Year Built", "Garage Yr Blt", "Yr Sold", "Gr Liv Area", "Sale Condition"],
]

display(verificacao)
pendente

# 5 · Corte por ausência e imputação do resto

Aqui entra a **Regra 1**: ausência acima de 5% não se imputa, se descarta — vale para
coluna e para registro.

O motivo é o que a imputação faz com a distribuição. Preencher 16,7% de uma coluna com a
mediana não completa o dado: cria um pico artificial no valor imputado, encolhe a
variância e faz um sexto da coluna passar a ser produto do método, não da observação.
Toda estatística calculada depois — correlação, distância, silhueta — herda esse pico
como se fosse informação da carteira.

E não adianta imputar melhor. A ausência de `Lot Frontage` é **explicável**: a taxa vai
de 12,8% no lote interno a 48,9% no de cul-de-sac, porque a testada não é registrada
quando a geometria não a define. Ausência explicada é a melhor situação possível para
imputar, e ainda assim são 490 valores inventados numa coluna de 2.930. A tabela abaixo é
a evidência de que a decisão foi tomada olhando o mecanismo, e não a régua.

## A ordem dentro do bloco

Primeiro a coluna, depois o registro. Registro que só passava dos 5% por causa de uma
coluna descartada deixa de passar quando ela sai — cortá-lo antes seria puni-lo duas
vezes pelo mesmo buraco.

O que sobra depois dos dois cortes está todo **abaixo de 1%**, e aí a imputação é segura:
nesse volume ela não desloca distribuição nenhuma. Cada coluna recebe a estratégia que o
mecanismo pede — zero onde o item não existe, moda no registro único de sistema elétrico.

In [ ]:
# Evidência: a ausência de `Lot Frontage` acompanha a geometria do lote.
geometria = (
    dados.assign(ausente=dados["Lot Frontage"].isna())
    .groupby("Lot Config", observed=True)
    .agg(imoveis=("ausente", "size"), sem_testada=("ausente", "sum"))
    .assign(pct=lambda t: (100 * t.sem_testada / t.imoveis).round(1))
    .sort_values("pct", ascending=False)
    .reset_index()
)
io.salvar_tabela(geometria, SECAO, "ausencia-testada-por-geometria", indice=False)
geometria

In [ ]:
#: A regra, escrita uma vez e aplicada aos dois eixos.
LIMIAR_AUSENCIA = preparacao.LIMIAR_AUSENCIA

# Eixo 1 — colunas. A proporção é medida sobre a ausência que sobrou depois do
# bloco 3, isto é, sobre a ausência real.
proporcao_coluna = dados.isna().mean()
fora_por_coluna = (
    proporcao_coluna[proporcao_coluna > LIMIAR_AUSENCIA]
    .mul(100)
    .round(2)
    .rename("pct_ausente")
    .to_frame()
    .assign(celulas=lambda t: (t.pct_ausente / 100 * len(dados)).round().astype(int))
    .reset_index(names="coluna")
)
dados = dados.drop(columns=fora_por_coluna.coluna)

# Eixo 2 — registros, já sobre as colunas que ficaram.
proporcao_registro = dados.isna().mean(axis=1)
fora_por_registro = dados.loc[
    proporcao_registro > LIMIAR_AUSENCIA, ["Order", "PID", "Neighborhood"]
].assign(
    campos_vazios=dados.loc[proporcao_registro > LIMIAR_AUSENCIA].isna().sum(axis=1),
    pct_vazio=(100 * proporcao_registro[proporcao_registro > LIMIAR_AUSENCIA]).round(2),
)
dados = dados[proporcao_registro <= LIMIAR_AUSENCIA]

io.salvar_tabela(fora_por_coluna, SECAO, "corte-por-ausencia-colunas", indice=False)
io.salvar_tabela(fora_por_registro, SECAO, "corte-por-ausencia-registros", indice=False)

registrar(
    "5 · Corte por ausência",
    "Descartar as colunas com mais de 5% de ausência real",
    "Imputar 16,7% de `Lot Frontage` produziria um sexto da coluna pelo próprio método, "
    "com pico artificial na mediana. Nenhuma das duas colunas está no espaço de "
    "atributos: `testada_lote` e `ano_garagem` são descritivas.",
    len(fora_por_coluna),
    unidade="colunas",
    acao="removido",
)
registrar(
    "5 · Corte por ausência",
    "Descartar os registros com mais de 5% dos campos vazios",
    "Registro que não descreve 5% do próprio cadastro seria completado, não imputado. "
    "O corte roda depois do de colunas, para não punir o registro por um buraco que já "
    "saiu da base.",
    len(fora_por_registro),
    unidade="registros",
    acao="removido",
)

display(fora_por_coluna)
fora_por_registro

In [ ]:
imputacoes = []

# Áreas e contagens de itens que simplesmente não existem no imóvel.
ZEROS = [
    "Mas Vnr Area",
    "BsmtFin SF 1",
    "BsmtFin SF 2",
    "Bsmt Unf SF",
    "Total Bsmt SF",
    "Bsmt Full Bath",
    "Bsmt Half Bath",
    "Garage Cars",
    "Garage Area",
]
for coluna in ZEROS:
    celulas = int(dados[coluna].isna().sum())
    dados[coluna] = dados[coluna].fillna(0)
    imputacoes.append(
        {
            "coluna": coluna,
            "estrategia": "zero (o item não existe)",
            "celulas": celulas,
            "pct_da_coluna": round(100 * celulas / len(dados), 2),
        }
    )

# Sistema elétrico: registro único, imputado pela moda.
moda = dados["Electrical"].mode().iloc[0]
celulas = celulas_eletrico = int(dados["Electrical"].isna().sum())
dados["Electrical"] = dados["Electrical"].fillna(moda)
imputacoes.append(
    {
        "coluna": "Electrical",
        "estrategia": f"moda ({moda})",
        "celulas": celulas,
        "pct_da_coluna": round(100 * celulas / len(dados), 2),
    }
)

imputacoes = pd.DataFrame(imputacoes).sort_values("celulas", ascending=False)
io.salvar_tabela(imputacoes, SECAO, "estrategias-de-imputacao", indice=False)

# Nenhuma imputação passa do limiar — é a Regra 1 conferida, não presumida.
maior = imputacoes.pct_da_coluna.max()
assert maior <= 100 * LIMIAR_AUSENCIA, f"imputação acima do limiar: {maior}%"

registrar(
    "5 · Imputação",
    "Preencher com zero as áreas e contagens de itens inexistentes",
    "Área de garagem ausente em imóvel sem garagem é zero, não desconhecida. Todas as "
    "colunas envolvidas estão abaixo de 0,8% de ausência, dentro do limiar da Regra 1.",
    int(imputacoes[imputacoes.estrategia.str.startswith("zero")].celulas.sum()),
    unidade="células",
    acao="imputado",
)
registrar(
    "5 · Imputação",
    f"Preencher `sistema_eletrico` pela moda (`{moda}`)",
    "Registro único ausente numa coluna em que 91,5% dos imóveis compartilham o mesmo sistema.",
    celulas_eletrico,
    unidade="células",
    acao="imputado",
)

print(f"Maior imputação: {maior}% de uma coluna — limiar da Regra 1 em {100 * LIMIAR_AUSENCIA}%")

print(f"Células vazias restantes: {pt(int(dados.isna().sum().sum()))}")
imputacoes

# 6 · Codificação das escalas ordinais

Vinte colunas guardam **ordem** como texto: `Ex > Gd > TA > Fa > Po` em dez colunas de
qualidade, e mais dez escalas próprias (exposição do porão, funcionalidade, formato do
lote, acesso pavimentado…). Tratá-las como nominal e aplicar *one-hot* destruiria a
ordem e multiplicaria a dimensionalidade sem ganho nenhum.

O mapa vem de `dic.ESCALAS_ORDINAIS`, e a convenção é uma só: **0 para ausência**. Um
imóvel sem porão recebe `qualidade_porao = 0`, fora da escala e abaixo do pior nível
existente (`Po = 1`) — que é a leitura correta, porque "não tem porão" é pior que "tem
porão em péssimo estado" para quem compara imóveis.

In [ ]:
amostra_antes = dados[list(dic.ESCALAS_ORDINAIS)].head(3).copy()

codificacao = []
for coluna, escala in dic.ESCALAS_ORDINAIS.items():
    niveis = sorted(escala, key=escala.get)
    fora_da_escala = int((~dados[coluna].isin(escala)).sum())
    dados[coluna] = dados[coluna].map(escala).fillna(0).astype("int8")
    codificacao.append(
        {
            "coluna": coluna,
            "escala": " < ".join(niveis),
            "niveis": len(escala),
            "registros_em_zero": fora_da_escala,
            "mediana_codificada": float(dados[coluna].median()),
        }
    )

codificacao = pd.DataFrame(codificacao).sort_values("registros_em_zero", ascending=False)
io.salvar_tabela(codificacao, SECAO, "codificacao-ordinal", indice=False)

registrar(
    "6 · Codificação",
    f"Mapear {len(dic.ESCALAS_ORDINAIS)} colunas ordinais para inteiro, com 0 para ausência",
    "`Ex > Gd > TA > Fa > Po` tem ordem semântica. One-hot destruiria a ordem e "
    "multiplicaria a dimensionalidade sem ganho.",
    len(dic.ESCALAS_ORDINAIS),
    unidade="colunas",
)

display(amostra_antes)
display(dados[list(dic.ESCALAS_ORDINAIS)].head(3))
codificacao

# 7 · Renomeação — de `Gr Liv Area` a `area_habitavel`

Os 82 nomes originais têm espaço no meio, o que obriga a usar colchetes em toda
referência e impede o acesso por atributo. O resultado deste projeto é lido por gente de
negócio, e `area_habitavel` diz mais do que `Gr Liv Area` para quem trabalha na
Meridiano.

O mapa completo está em `dic.RENOMEAR` — **fonte única da verdade** sobre nomes. A
verificação abaixo garante que nenhuma coluna presente ficou sem nome curado. O contrário
não é exigido: o mapa descreve as 82 colunas da origem, e duas delas já saíram no corte de
ausência do bloco 5.

In [ ]:
curada = dados.rename(columns=dic.RENOMEAR)

# Toda coluna presente precisa ter nome curado. O contrário não vale: o mapa descreve
# as 82 colunas da origem, e duas delas já saíram no corte de ausência.
nao_mapeadas = [coluna for coluna in dados.columns if coluna not in dic.RENOMEAR]
ja_removidas = [coluna for coluna in dic.RENOMEAR if coluna not in dados.columns]
assert not nao_mapeadas, f"colunas sem nome curado: {nao_mapeadas}"

registrar(
    "7 · Renomeação",
    f"Renomear as {len(curada.columns)} colunas para `snake_case` em português",
    "Nome com espaço no meio obriga colchete em toda referência e impede acesso por "
    "atributo. A base curada é lida por gente de negócio.",
    len(curada.columns),
    unidade="colunas",
)

print(f"{len(curada.columns)} colunas renomeadas · nenhuma sem correspondência")
print(f"Fora do mapa por já terem saído no bloco 5: {', '.join(ja_removidas) or '—'}")
curada.head(3)

# 8 · Atributos derivados

Nove colunas novas, construídas a partir das que já estão limpas. Duas razões justificam
cada grupo:

* **somar o que mede a mesma coisa.** Cinco colunas de varanda (deck, aberta, fechada,
  de três estações, de tela) são quase todas zero quando lidas isoladamente — nenhuma
  tem poder de separação. Somadas em `area_varandas`, viram um atributo com variância
  útil. O mesmo vale para `banheiros_total`, que conta o lavabo como meio banheiro e
  soma o porão, do jeito que o mercado conta;
* **trocar ano por idade.** `ano_construcao = 1960` não diz nada sozinho: o que separa
  imóveis é a **idade na data da venda**. `idade_imovel` e `anos_desde_reforma` fazem
  essa conta e são recortadas em zero — imóvel vendido na planta, com venda registrada
  antes da conclusão, ficaria com idade negativa.

As quatro flags (`tem_porao`, `tem_garagem`, `tem_lareira`, `tem_segundo_pavimento`)
guardam a informação binária que a área sozinha esconde depois de padronizada.

`preco_por_pe2` é derivada do **rótulo reservado** e por isso não entra em nenhuma etapa
de modelagem — fica na base curada apenas para a avaliação *a posteriori*.

In [ ]:
curada["area_varandas"] = (
    curada["area_deck"]
    + curada["area_varanda_aberta"]
    + curada["area_varanda_fechada"]
    + curada["area_varanda_3estacoes"]
    + curada["area_varanda_tela"]
)
curada["banheiros_total"] = (
    curada["banheiros_completos"]
    + 0.5 * curada["lavabos"]
    + curada["banheiros_completos_porao"]
    + 0.5 * curada["lavabos_porao"]
)
curada["idade_imovel"] = curada["ano_venda"] - curada["ano_construcao"]
curada["anos_desde_reforma"] = curada["ano_venda"] - curada["ano_reforma"]
curada["tem_porao"] = (curada["area_porao"] > 0).astype("int8")
curada["tem_garagem"] = (curada["area_garagem"] > 0).astype("int8")
curada["tem_lareira"] = (curada["lareiras"] > 0).astype("int8")
curada["tem_segundo_pavimento"] = (curada["area_pavimento_superior"] > 0).astype("int8")

# Idade negativa aparece em imóvel vendido na planta — venda registrada no ano anterior
# à conclusão. O piso em zero mantém o registro sem inventar uma idade impossível.
negativas = int((curada["idade_imovel"] < 0).sum() + (curada["anos_desde_reforma"] < 0).sum())
curada["idade_imovel"] = curada["idade_imovel"].clip(lower=0)
curada["anos_desde_reforma"] = curada["anos_desde_reforma"].clip(lower=0)

# Derivada do rótulo reservado: entra apenas na avaliação a posteriori.
curada["preco_por_pe2"] = curada["preco_venda"] / curada["area_habitavel"]

DERIVADAS = [
    "area_varandas",
    "banheiros_total",
    "idade_imovel",
    "anos_desde_reforma",
    "tem_porao",
    "tem_garagem",
    "tem_lareira",
    "tem_segundo_pavimento",
    "preco_por_pe2",
]

registrar(
    "8 · Derivadas",
    f"Derivar {len(DERIVADAS) - 1} atributos de modelagem mais `preco_por_pe2`, reservado",
    "Cinco colunas de varanda medem a mesma coisa e, separadas, são quase todas zero. "
    "Somadas viram um atributo com variância útil. Ano de construção só separa imóveis "
    "depois de virar idade na data da venda.",
    len(DERIVADAS),
    unidade="colunas",
)
if negativas:
    registrar(
        "8 · Derivadas",
        "Recortar em zero as idades negativas",
        "Imóvel vendido na planta tem a venda registrada antes da conclusão da obra. O "
        "piso em zero mantém o registro sem inventar idade impossível.",
        negativas,
        unidade="registros",
    )

resumo_derivadas = curada[DERIVADAS].describe().T.round(2)
io.salvar_tabela(
    resumo_derivadas.reset_index(names="variavel"), SECAO, "atributos-derivados", indice=False
)
resumo_derivadas

# 9 · Registros removidos

Cinco imóveis saem da base, e a decisão **não é nossa**: é recomendação explícita do
autor do conjunto de dados.

> *"I would recommend removing any houses with more than 4000 square feet from the data
> set (which eliminates these five unusual observations) before assigning it to
> students."*
> — De Cock (2011), *Journal of Statistics Education* 19(3), seção 4

Três dos cinco são vendas `Partial` — imóveis registrados antes da conclusão da obra,
com preço que não corresponde ao produto entregue. Um deles vende 5.642 pés² por
US$ 160.000, menos que a mediana da carteira: é venda parcial, não imóvel barato.

A remoção vem **por último**, depois da imputação, para que nenhuma mediana de bairro
tenha sido calculada sobre uma base já podada.

In [ ]:
LIMITE_AREA = 4000

removidos = curada.loc[
    curada["area_habitavel"] > LIMITE_AREA,
    [
        "ordem",
        "pid",
        "bairro",
        "area_habitavel",
        "qualidade_geral",
        "preco_venda",
        "condicao_venda",
    ],
].copy()
io.salvar_tabela(removidos, SECAO, "registros-removidos", indice=False)

antes_da_remocao = len(curada)
curada = curada[curada["area_habitavel"] <= LIMITE_AREA].reset_index(drop=True)

registrar(
    "9 · Remoção",
    f"Remover os imóveis com área habitável acima de {pt(LIMITE_AREA)} pés²",
    "Recomendação explícita do autor da base (De Cock, 2011, seção 4): são vendas "
    "parciais de imóveis atípicos que distorcem qualquer análise. Decisão citável, "
    "não arbitrária.",
    antes_da_remocao - len(curada),
    unidade="registros",
    acao="removido",
)

print(f"{pt(antes_da_remocao)} → {pt(len(curada))} registros")
removidos

# 10 · Depuração da carteira

Aqui entra a **Regra 2**: sai da carteira o que não descreve um imóvel residencial
comparável. Quatro cortes, cada um medido sobre a base como ela está quando o corte roda
— a ordem muda os números, e por isso ela está declarada.

## 10.1 · Valores extremos, a 3×IQR e não a 1,5×IQR

O critério de Tukey é 1,5×IQR, e ele marca **635 registros, 21,7% da carteira**. Aplicá-lo
seria remover o segmento de alto padrão que a segmentação existe para encontrar: um imóvel
de 3.500 pés² com nota 10 de qualidade não é erro de cadastro, é o produto mais caro da
Meridiano.

O corte usa **3×IQR**, que alcança só a anomalia. É a diferença entre tratar o extremo e
tratar a cauda — e a cauda, num problema de segmentação, é justamente onde mora um dos
grupos.

A tabela compara os dois critérios variável a variável, para que a escolha seja auditável
e não uma preferência.

In [ ]:
NUMERICAS_CURADAS = [c for c in dic.NUMERICAS + dic.ORDINAIS if c in curada.columns]
FATOR_EXTREMO = preparacao.FATOR_EXTREMO

linhas = []
fora_da_cerca = pd.Series(False, index=curada.index)
for coluna in NUMERICAS_CURADAS:
    serie = curada[coluna]
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    tukey = est.outliers_iqr(serie)
    extremo = (serie < q1 - FATOR_EXTREMO * iqr) | (serie > q3 + FATOR_EXTREMO * iqr)
    fora_da_cerca |= extremo
    linhas.append(
        {
            "variavel": coluna,
            "tukey_1_5": int(tukey.sum()),
            "pct_tukey": round(100 * tukey.mean(), 2),
            "extremos_3_0": int(extremo.sum()),
            "pct_extremos": round(100 * extremo.mean(), 2),
            "limite_superior_3_0": q3 + FATOR_EXTREMO * iqr,
            "maior_valor": serie.max(),
        }
    )

discrepantes = pd.DataFrame(linhas).sort_values("pct_tukey", ascending=False)

comparacao = discrepantes.melt(
    id_vars="variavel",
    value_vars=["pct_tukey", "pct_extremos"],
    var_name="criterio",
    value_name="percentual",
).replace({"pct_tukey": "Tukey · 1,5×IQR", "pct_extremos": "Extremo · 3×IQR"})

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(
    data=comparacao,
    x="percentual",
    y="variavel",
    hue="criterio",
    palette={"Tukey · 1,5×IQR": config.CINZA, "Extremo · 3×IQR": ALERTA},
    ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f%%", fontsize=7, padding=2)
ax.set_xlabel("% de registros fora da cerca")
ax.set_ylabel("")
ax.set_title("Dois critérios de discrepância — só o de 3×IQR é aplicado")
ax.legend(title="Critério")
ax.margins(x=0.16)

salvar(fig, "discrepantes-dois-criterios", discrepantes)

# A contagem por variável soma ocorrências; o corte é por registro, e um registro
# extremo em três variáveis sai uma vez só.
tukey_registros = pd.DataFrame(
    {c: est.outliers_iqr(curada[c]) for c in NUMERICAS_CURADAS}
).any(axis=1)

antes_extremos = len(curada)
curada = curada[~fora_da_cerca]

registrar(
    "10 · Depuração",
    f"Remover os valores extremos, além de {FATOR_EXTREMO:.0f}×IQR em qualquer atributo",
    f"A cerca de Tukey marcaria {int(tukey_registros.sum())} registros "
    f"({100 * tukey_registros.mean():.1f}%) e levaria junto o segmento de alto padrão "
    "que a segmentação precisa isolar. O fator 3 alcança só a anomalia.",
    antes_extremos - len(curada),
    unidade="registros",
    acao="removido",
)
discrepantes

## 10.2 · Vendas que não são normais

`condicao_venda` descreve **como** a transação aconteceu. Cinco das seis categorias
descrevem situação em que o preço não corresponde ao produto:

| Categoria | O que é |
|:---|:---|
| `Partial` | Imóvel entregue antes da conclusão do cadastro — o preço é o da obra incompleta |
| `Abnorml` | Execução, retomada, venda forçada |
| `Family` | Venda entre familiares, com preço acordado fora do mercado |
| `Alloca` | Duas matrículas ligadas vendidas juntas, com o valor repartido entre elas |
| `AdjLand` | Venda de terreno adjacente, não de imóvel |

Sem esse corte, `preco_venda` mede duas coisas ao mesmo tempo — o imóvel e a circunstância
da venda — e o rótulo reservado deixa de servir como validação externa dos segmentos.

Duas consequências ficam registradas, porque são reais: é a **maior remoção do notebook**,
e ela esvazia a análise de sensibilidade que `02b-sensibilidade-escala` fazia sobre essa
coluna. A coluna continua na base, agora constante em `Normal`, porque o esquema da matriz
de modelagem depende dela.

In [ ]:
composicao = (
    curada["condicao_venda"]
    .value_counts()
    .rename("registros")
    .to_frame()
    .assign(pct=lambda t: (100 * t.registros / len(curada)).round(2))
    .reset_index(names="condicao_venda")
)
io.salvar_tabela(composicao, SECAO, "composicao-da-condicao-de-venda", indice=False)

antes_vendas = len(curada)
curada = curada[curada["condicao_venda"] == "Normal"]

registrar(
    "10 · Depuração",
    "Remover as vendas não normais",
    "Leilão, venda entre familiares e imóvel entregue antes da conclusão do cadastro têm "
    "preço que não descreve o produto. Sem elas, `preco_venda` volta a medir só o imóvel.",
    antes_vendas - len(curada),
    unidade="registros",
    acao="removido",
)
composicao

## 10.3 · Cadastros implausíveis

Os imóveis sem dormitório acima do solo **não são erro de cadastro** — a verificação está
no bloco 4: são *split foyer*, *split level* e duplex, com o dormitório no porão, e
`dormitorios` conta apenas o pavimento acima do solo. O mesmo vale para o único sem
banheiro completo, que tem dois lavabos acima e dois no porão.

São imóveis reais, e mesmo assim saem. O motivo não é qualidade do dado, é
**comparabilidade**: `dormitorios` e `banheiros_total` estão no espaço de atributos, e um
imóvel com zero em qualquer um dos dois ocupa uma região que nenhum outro ocupa. Em
algoritmo baseado em distância isso vira grupo de um elemento, que não diz nada sobre a
carteira e consome um dos cinco grupos que a diretoria financia.

Parte deles já saiu nos cortes anteriores; a contagem abaixo é sobre o que restou.

In [ ]:
implausivel = (curada["dormitorios"] == 0) | (
    curada["banheiros_completos"] + curada["banheiros_completos_porao"] == 0
)
cadastros = curada.loc[
    implausivel,
    [
        "ordem",
        "estilo",
        "tipo_construcao",
        "area_habitavel",
        "dormitorios",
        "banheiros_completos",
        "banheiros_completos_porao",
        "preco_venda",
    ],
]
io.salvar_tabela(cadastros, SECAO, "cadastros-implausiveis", indice=False)

curada = curada[~implausivel]

registrar(
    "10 · Depuração",
    "Remover os cadastros sem dormitório acima do solo e sem banheiro completo",
    "São imóveis reais — o dormitório fica no porão em split foyer, split level e duplex "
    "—, mas com zero em duas variáveis do espaço de atributos ocupam uma região que "
    "nenhum outro imóvel ocupa e viram grupo de um elemento.",
    len(cadastros),
    unidade="registros",
    acao="removido",
)
cadastros

## 10.4 · Colunas que deixaram de informar

Coluna em que 99% dos imóveis dizem a mesma coisa não separa ninguém: a variância é quase
nula e, depois da padronização, ela entra como ruído com o mesmo peso de uma variável que
discrimina.

O corte roda **por último**, depois das remoções de registro, porque a concentração muda
com elas — `material_telhado` só passa dos 99% na carteira já depurada.

Uma exceção declarada: as colunas do **esqueleto** da base ficam mesmo passando do limiar.
`condicao_venda` é 100% `Normal` justamente porque o corte 10.2 a tornou constante, e a
matriz de modelagem depende dela para existir. Remover uma coluna porque uma decisão
anterior a esvaziou seria apagar o rastro dessa decisão.

In [ ]:
PROTEGIDAS = preparacao.COLUNAS_PROTEGIDAS

concentracao = []
for coluna in curada.columns:
    preenchida = curada[coluna].dropna()
    if not len(preenchida):
        continue
    frequencia = preenchida.value_counts(normalize=True)
    concentracao.append(
        {
            "coluna": coluna,
            "distintos": int(preenchida.nunique()),
            "valor_dominante": frequencia.index[0],
            "pct_dominante": round(100 * frequencia.iloc[0], 2),
            "protegida": coluna in PROTEGIDAS,
        }
    )

concentracao = pd.DataFrame(concentracao).sort_values("pct_dominante", ascending=False)
sem_informacao = concentracao.query("pct_dominante >= @preparacao.LIMIAR_CONCENTRACAO")
io.salvar_tabela(sem_informacao, SECAO, "colunas-sem-informacao", indice=False)

descartar = sem_informacao.query("not protegida").coluna.tolist()
curada = curada.drop(columns=descartar)

# Último corte da carteira: o índice volta a ser sequencial, para que a base gravada
# não carregue a memória de quem saiu.
curada = curada.reset_index(drop=True)

registrar(
    "10 · Depuração",
    "Descartar as colunas concentradas acima de 99% num único valor",
    "Não separam ninguém e, depois da padronização, entram como ruído com o mesmo peso de "
    "uma variável que discrimina. As colunas do esqueleto ficam: `condicao_venda` virou "
    "constante por causa do corte anterior, e a matriz de modelagem depende dela.",
    len(descartar),
    unidade="colunas",
    acao="removido",
)

print(f"Descartadas: {', '.join(descartar)}")
sem_informacao

# Conferência da base curada

Três verificações antes de gravar. Nenhuma delas é opcional: é o que separa "rodou sem
erro" de "está correto".

1. **Nulo zerado** — nenhuma célula vazia sobrou, e nenhuma foi preenchida sem que o
   bloco 3 ou o 5 tenha registrado como;
2. **Plausibilidade do domínio** — as oito regras de `qualidade.valores_implausiveis`
   voltam a rodar sobre a base curada, e agora **todas** precisam marcar zero: as duas que
   antes acusavam característica legítima (sem dormitório acima do solo, sem banheiro
   completo) foram tratadas em 10.3. A regra da garagem aparece como *não aplicável*,
   porque `ano_garagem` saiu no corte de ausência — checagem que deixou de rodar continua
   no relatório, em vez de sumir dele;
3. **Assimetria** — quais numéricas continuam assimétricas o bastante para justificar
   `log1p` antes de qualquer distância. Este é o insumo direto de `dic.ASSIMETRICAS`, e
   a transformação em si pertence ao pré-processador da modelagem, não à base curada:
   ela é gravada em **unidades originais**, para que o negócio a leia em pés quadrados e
   em dólares.

A leitura da assimetria muda com a depuração, e é uma verificação a fazer, não um número
a repetir: `area_terreno` chegava a 13,2 na base bruta por causa dos lotes de mais de
200.000 pés², que o corte de extremos removeu. As variáveis que continuarem acima do
limiar são as que justificam `log1p`; as ordinais ficam de fora por natureza, porque
`log1p` sobre nota de 0 a 5 destrói a leitura de passo constante que a escala tem.

Se a lista divergir de `dic.ASSIMETRICAS`, é o dicionário que precisa ser revisto — a
declaração lá foi medida sobre a base anterior a estes cortes.

In [ ]:
nulos = int(curada.isna().sum().sum())
print(f"Células vazias na base curada: {nulos}")
assert nulos == 0, "a base curada não pode ter célula vazia"

plausibilidade = qualidade.valores_implausiveis(curada)
io.salvar_tabela(plausibilidade.reset_index(), SECAO, "plausibilidade-apos-ajustes", indice=False)

verificadas = plausibilidade.query("situacao == 'verificada'")
assert (verificadas.registros == 0).all(), "regra de plausibilidade ainda violada"
nao_aplicaveis = len(plausibilidade) - len(verificadas)
print(f"{len(verificadas)} regras verificadas, todas em zero · {nao_aplicaveis} não aplicável")
plausibilidade

In [ ]:
assimetria = (
    curada[NUMERICAS_CURADAS]
    .skew()
    .rename("assimetria")
    .to_frame()
    .assign(
        acima_do_limiar=lambda t: t.assimetria.abs() >= dic.LIMIAR_ASSIMETRIA,
        declarada_em_dicionario=lambda t: t.index.isin(dic.ASSIMETRICAS),
    )
    .sort_values("assimetria", ascending=False)
    .reset_index(names="variavel")
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(
    data=assimetria,
    x="assimetria",
    y="variavel",
    hue="acima_do_limiar",
    palette={True: ALERTA, False: BARRA},
    dodge=False,
    ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f", fontsize=8, padding=3)
ax.axvline(dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.axvline(-dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.set_xlabel(f"Assimetria — limiar do projeto em ±{dic.LIMIAR_ASSIMETRIA}")
ax.set_ylabel("")
ax.set_title("Assimetria das variáveis do espaço de atributos, na base curada")
ax.legend(title=f"|assimetria| ≥ {dic.LIMIAR_ASSIMETRIA}")
ax.margins(x=0.12)

salvar(fig, "assimetria-apos-ajustes", assimetria)

# As ordinais passam do limiar por construção da escala e não recebem `log1p`; a
# conferência é sobre as contínuas.
CONTINUAS = [c for c in dic.NUMERICAS if c in curada.columns]
candidatas = assimetria.query("acima_do_limiar and variavel in @CONTINUAS").variavel.tolist()
if sorted(candidatas) == sorted(dic.ASSIMETRICAS):
    print(f"`dic.ASSIMETRICAS` confere com a medição: {', '.join(sorted(candidatas))}")
else:
    print(f"Revisar `dic.ASSIMETRICAS`: medido {candidatas}, declarado {dic.ASSIMETRICAS}")

assimetria

## Equivalência com o pipeline

A base curada tem **dois produtores**: este notebook, que narra cada decisão, e
`src/data/preparacao.construir_base_curada()`, que o pipeline executa sem supervisão
(`uv run invoke preparacao`). Os dois precisam produzir exatamente a mesma tabela — se
divergirem, a documentação descreve uma base e o modelo consome outra.

A comparação abaixo é a prova. Qualquer célula divergente aparece na tabela; o esperado
é que ela saia vazia.

In [ ]:
referencia = preparacao.construir_base_curada(salvar=False)

print(f"notebook: {curada.shape}   ·   módulo: {referencia.shape}")
assert list(curada.columns) == list(referencia.columns), "ordem ou conjunto de colunas difere"

linhas = []
for coluna in curada.columns:
    esquerda, direita = curada[coluna], referencia[coluna]
    divergem = ~((esquerda == direita) | (esquerda.isna() & direita.isna()))
    if divergem.any() or esquerda.dtype != direita.dtype:
        linhas.append(
            {
                "coluna": coluna,
                "celulas_divergentes": int(divergem.sum()),
                "tipo_notebook": str(esquerda.dtype),
                "tipo_modulo": str(direita.dtype),
            }
        )

divergencias = pd.DataFrame(
    linhas, columns=["coluna", "celulas_divergentes", "tipo_notebook", "tipo_modulo"]
)
io.salvar_tabela(divergencias, SECAO, "equivalencia-com-o-pipeline", indice=False)

if divergencias.empty:
    print("Sem divergência: o notebook e o pipeline produzem a mesma base curada.")
divergencias

# A base curada

O que sai deste notebook:

* **`data/processed/imoveis_curados.parquet`** — a base curada, em unidades originais.
  É a entrada da matriz de modelagem (`src/data/preparacao.construir_matriz_modelagem`),
  da análise exploratória sobre dado tratado e da aplicação de implantação;
* **`data/processed/manifesto_curadoria.json`** — o que entrou, o que saiu e quantos
  ajustes, no mesmo formato do `manifesto_carga.json` gravado pela ingestão;
* **`reports/tables/ajustes-dados/registro-de-ajustes.csv`** — o registro completo, que
  a documentação publica.

O parquet é gravado **sem transformação de escala**: nada de `log1p`, nada de
padronização. Essas operações dependem do algoritmo e vivem no pré-processador
(`preparacao.construir_preprocessador`), que é ajustado dentro da validação. Guardar dado
padronizado em disco é o caminho mais curto para vazamento entre etapas.

In [ ]:
config.garantir_diretorios()
curada.to_parquet(config.BASE_CURADA, index=False)

registro = pd.DataFrame(AJUSTES)
io.salvar_tabela(registro, SECAO, "registro-de-ajustes", indice=False)

manifesto = {
    "origem": origem.relative_to(RAIZ).as_posix(),
    "destino": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
    "linhas_entrada": int(LINHAS_ENTRADA),
    "colunas_entrada": int(COLUNAS_ENTRADA),
    "linhas_saida": int(len(curada)),
    "colunas_saida": int(curada.shape[1]),
    "registros_removidos": int(LINHAS_ENTRADA - len(curada)),
    "pct_registros_removidos": round(100 * (LINHAS_ENTRADA - len(curada)) / LINHAS_ENTRADA, 2),
    "limiar_ausencia": LIMIAR_AUSENCIA,
    "fator_extremo": FATOR_EXTREMO,
    "colunas_derivadas": DERIVADAS,
    "celulas_vazias_entrada": int(bruto.isna().sum().sum()),
    "celulas_vazias_saida": int(curada.isna().sum().sum()),
    "ajustes": int(len(registro)),
    "ajustes_por_acao": registro.acao.value_counts().to_dict(),
    "equivalente_ao_pipeline": bool(divergencias.empty),
}
io.salvar_json(manifesto, config.DATA_PROCESSED / "manifesto_curadoria.json")

print(f"Base curada gravada em {config.BASE_CURADA.relative_to(RAIZ)}")
print(
    f"{pt(len(curada))} registros × {curada.shape[1]} colunas · {len(registro)} ajustes registrados"
)
pd.Series(manifesto).to_frame("manifesto de curadoria")

In [ ]:
# O funil: de onde saiu cada registro que não chegou à base curada.
removidos = registro.query("acao == 'removido' and unidade == 'registros'")
etapas = [{"etapa": "Base bruta", "registros": LINHAS_ENTRADA}]
restante = LINHAS_ENTRADA
for _, linha in removidos.iterrows():
    restante -= linha.afetados
    etapas.append({"etapa": linha.ajuste, "registros": restante})
funil = pd.DataFrame(etapas)
funil["removidos"] = funil.registros.shift(1).sub(funil.registros).fillna(0).astype(int)
funil["pct_do_total"] = (100 * funil.registros / LINHAS_ENTRADA).round(1)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=funil, x="registros", y="etapa", color=BARRA, ax=ax)
for grupo in ax.containers:
    ax.bar_label(
        grupo,
        labels=[
            f"{pt(int(r))}  ({p:.1f}%)" + (f"  −{pt(int(d))}" if d else "")
            for r, p, d in zip(funil.registros, funil.pct_do_total, funil.removidos)
        ],
        fontsize=9,
        padding=4,
    )
ax.set_xlabel("Registros restantes")
ax.set_ylabel("")
ax.set_title(f"Funil da curadoria — {pt(LINHAS_ENTRADA)} → {pt(len(curada))} imóveis")
ax.margins(x=0.22)

salvar(fig, "funil-da-curadoria", funil)
funil

In [ ]:
registro

In [ ]:
# Amostra do que a próxima fase recebe: as variáveis do espaço de atributos, já limpas,
# codificadas e derivadas, em unidades originais.
curada[dic.ESPACO_MISTO + dic.BINARIAS].head(10)

# Próximo passo

A base curada está gravada e é idêntica à que o pipeline produz. A partir daqui:

* **`src/data/preparacao.construir_matriz_modelagem`** recorta as variáveis do espaço de
  atributos e grava `data/processed/matriz_modelagem.parquet` — sem o rótulo reservado,
  sem o bairro;
* **[`02b-sensibilidade-escala`](02b-sensibilidade-escala.ipynb)** compara os
  escaladores sobre essa matriz e mostra o efeito de cada um na estrutura de vizinhança;
* **[`03-clusterizacao-particional-hierarquica`](03-clusterizacao-particional-hierarquica.ipynb)**
  entra na fase 4 do CRISP-DM.

As decisões deste notebook estão publicadas em `docs/preparacao.md`, e o registro de
ajustes é a tabela que aquela página cita.